# Lightweight Fine-Tuning Project

TODO: In this cell, describe your choices for each of the following

* PEFT technique: LoRA
* Model: RoBERTa
* Evaluation approach: Compute evaluation metrics: Accuracy, Recall, Precision and F1 score.
* Fine-tuning dataset: jahjinx/IMDb_movie_reviews

## Loading and Evaluating a Foundation Model

TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset.

In [48]:
! pip install -q datasets==2.15.0

In [49]:
from datasets import load_dataset

# Load the IMDb dataset from the `jahjinx` repository
my_dataset = load_dataset("jahjinx/IMDb_movie_reviews")

# Access all splits: train, validation, and test
train_dataset = my_dataset["train"]#.shuffle(seed=42).select(range(10000))
validation_dataset = my_dataset["validation"]
test_dataset = my_dataset["test"]#.shuffle(seed=42).select(range(2000))

In [50]:
test_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 10000
})

In [51]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Pre-trained RoBERTa for sequence classification
model_name = "roberta-base"

my_tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model for binary classification
base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [52]:
# Tokenize our data
def get_tokens(data, tokenizer=my_tokenizer):
    return tokenizer(data['text'], truncation = True, padding=True)

tokenized_train = train_dataset.map(get_tokens, batched = 64)
tokenized_val = validation_dataset.map(get_tokens, batched = 64)
tokenized_test = test_dataset.map(get_tokens, batched = 64)


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [53]:
# Set format for  the test dataset
test_data = tokenized_test.rename_column("label", "labels")
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [55]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# Define compute_metrics function to calculate metrics of interest: Accuracy, Recall, Precision and F1 score.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions)
    recall = recall_score(labels, predictions)
    f1 = f1_score(labels, predictions)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

In [56]:
from transformers import Trainer, TrainingArguments

# Define Trainer arguments
training_args_base = TrainingArguments(
                                        output_dir="./results",
                                        per_device_eval_batch_size=32,  # Batch size for evaluation
                                        do_train=False,  # Skip training since we are just evaluating the base model here.
                                        logging_dir = "./logs",
                                        logging_steps = 10 ,
                                        seed = 42,
                                        report_to="none"
                                    )

# Initialize the Trainer
trainer_base = Trainer(
                          model = base_model,
                          args = training_args_base,
                          tokenizer = my_tokenizer,
                          compute_metrics = compute_metrics,
                      )

<ipython-input-56-ac87bcdc7e37>:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_base = Trainer(


In [58]:
# Eliminate randomness to get consistent results
import random
import numpy as np
import torch

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)  # Ensure reproducibility across GPUs

In [60]:
# Use the Trainer for prediction on the test dataset
results = trainer_base.predict(test_data)

# Extract metrics
metrics = results.metrics

print("Metrics for the base RoBERTa model:")
print('=='*20)
print(f"Accuracy of the base RoBERTa model: {metrics['test_accuracy']:.4f}")
print('--'*20)
print(f"Precision of the base RoBERTa model: {metrics['test_precision']:.4f}")
print('--'*20)
print(f"Recall of the base RoBERTa model: {metrics['test_recall']:.4f}")
print('--'*20)
print(f"F1 Score of the base RoBERTa model: {metrics['test_f1']:.4f}")
print('--'*20)

Metrics for the base RoBERTa model:
Accuracy of the base RoBERTa model: 0.5050
----------------------------------------
Precision of the base RoBERTa model: 0.5004
----------------------------------------
Recall of the base RoBERTa model: 0.8358
----------------------------------------
F1 Score of the base RoBERTa model: 0.6260
----------------------------------------


**Note:**

The base RoBERTa model achieved and accuray of only 0.5050, meaning it performed not better than random chance of the classification task. The model has a pretty high Recall, meaning it is biased towards positive class.

## Performing Parameter-Efficient Fine-Tuning

TODO: In the cells below, create a PEFT model from your loaded model, run a training loop, and save the PEFT model weights.

In [42]:
from peft import LoraConfig, get_peft_model

# Make the LoRA configuration
lora_config = LoraConfig(
                    task_type="SEQ_CLS",
                    r=8,
                    lora_alpha=32,
                    target_modules=["query", "value"],
                    lora_dropout=0.01,
                    bias = "none"
                    )

# Create the LoRA model
lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066


In [43]:
train_data = tokenized_train.rename_column("label", "labels")
val_data = tokenized_val.rename_column("label", "labels")

# Set the format for the training and evaluation data
train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_data.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [61]:
from transformers import Trainer, TrainingArguments

# Define the training arguments
training_args = TrainingArguments(
                                  output_dir="./results",
                                  do_eval=True,
                                  evaluation_strategy="epoch",
                                  save_strategy="epoch",
                                  logging_dir="./logs",
                                  per_device_train_batch_size=8,
                                  per_device_eval_batch_size=8,
                                  num_train_epochs=3,
                                  weight_decay=0.01,
                                  logging_steps=1500,
                                  save_total_limit=2,
                                  seed=42,
                                  report_to="none"
                                )

# Define the Trainer
trainer = Trainer(
                    model = lora_model,
                    args = training_args,
                    train_dataset = train_data,
                    eval_dataset = val_data,
                    compute_metrics = compute_metrics
                )

# Train the model
trainer.train()

# Save the LoRA weights
lora_model.save_pretrained("lora_roberta")

print("LoRA model saved...")

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.200600,0.186884,0.949250,0.941613,0.957243,0.949364
2,0.175100,0.198109,0.951750,0.948078,0.955231,0.951641
3,0.169900,0.184940,0.953500,0.948259,0.958753,0.953477


LoRA model saved...


## Performing Inference with a PEFT Model

TODO: In the cells below, load the saved PEFT model weights and evaluate the performance of the trained PEFT model. Be sure to compare the results to the results from prior to fine-tuning.

In [62]:
from peft import AutoPeftModelForSequenceClassification

# Load the LoRA model
lora_model = AutoPeftModelForSequenceClassification.from_pretrained("lora_roberta")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [63]:
# Use the Trainer for prediction on the test dataset
results1 = trainer.predict(test_data)

# Extract metrics
metrics1 = results1.metrics

print("Metrics for the LoRA model(RoBERTa):")
print('=='*20)
print(f"Accuracy of the LoRA model(RoBERTa) on test set: {metrics1['test_accuracy']:.4f}")
print('--'*20)
print(f"Precision of the LoRA model(RoBERTa) on test set: {metrics1['test_precision']:.4f}")
print('--'*20)
print(f"Recall of the LoRA model(RoBERTa) on test set: {metrics1['test_recall']:.4f}")
print('--'*20)
print(f"F1 Score of the LoRA model(RoBERTa) on test set: {metrics1['test_f1']:.4f}")
print('--'*20)

Metrics for the LoRA model(RoBERTa):
Accuracy of the LoRA model(RoBERTa) on test set: 0.9502
----------------------------------------
Precision of the LoRA model(RoBERTa) on test set: 0.9447
----------------------------------------
Recall of the LoRA model(RoBERTa) on test set: 0.9554
----------------------------------------
F1 Score of the LoRA model(RoBERTa) on test set: 0.9500
----------------------------------------


**Note**:

After fine-tuning using the LoRA technique, the model (RoBERTa) performed way better achieving Accuracy of 95% on the test set. The high value of other metrics (Precision, Recall and F1 score) testify that the model is a good one for achieving the task.